In [1]:
# Import libraries
import csv

import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42

In [2]:
# Set file paths
dataset = 'model/Greeting_keypoint/keypoint.csv'
model_save_path = 'model/Greeting_keypoint/keypoint_classifier.keras'
tflite_save_path = 'model/Greeting_keypoint/keypoint_classifier.tflite'

In [3]:
# Set class count
NUM_CLASSES = 15

In [4]:
# Load preprocessed features
# The CSV already contains normalized 42-value landmark vectors, so we load them directly.
X_dataset = np.loadtxt(dataset, delimiter=',', dtype='float32', usecols=list(range(1, (21 * 2) + 1)))

In [5]:
# Load labels
y_dataset = np.loadtxt(dataset, delimiter=',', dtype='int32', usecols=(0))

In [6]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X_dataset, y_dataset, train_size=0.75, random_state=RANDOM_SEED)

In [7]:
# Build classifier
# Input expects the preprocessed 42-value feature vector from the CSV.
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(42,)),  # 21*2 = 42
    
    tf.keras.layers.BatchNormalization(),
    
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    
    tf.keras.layers.Dense(32, activation='relu'),
    
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
])

In [8]:
# Show model summary
model.summary()  # tf.keras.utils.plot_model(model, show_shapes=True)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ batch_normalization             │ (None, 42)             │           168 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         5,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 15)             │           495 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,271 (67.46 KB)

 Trainable params: 16,803 (65.64 KB)

 Non-trainable params: 468 (1.83 KB)

In [9]:
# Set callbacks
# Model checkpoint callback
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    model_save_path, verbose=1, save_weights_only=False)
# Callback for early stopping
es_callback = tf.keras.callbacks.EarlyStopping(patience=20, verbose=1)

In [10]:
# Compile model
# Model compilation
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [11]:
# Train model
model.fit(
    X_train,
    y_train,
    epochs=1000,
    batch_size=128,
    validation_data=(X_test, y_test),
    callbacks=[cp_callback, es_callback]
)

Epoch 1/1000
56/62 ━━━━━━━━━━━━━━━━━━━━ 0s 919us/step - accuracy: 0.2521 - loss: 2.3895
Epoch 1: saving model to model/Greeting_keypoint/keypoint_classifier.keras
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.2737 - loss: 2.3239 - val_accuracy: 0.4550 - val_loss: 2.1827
Epoch 2/1000
 1/62 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7578 - loss: 0.9450
Epoch 2: saving model to model/Greeting_keypoint/keypoint_classifier.keras
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7867 - loss: 0.8237 - val_accuracy: 0.6739 - val_loss: 1.4837
Epoch 3/1000
 1/62 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8594 - loss: 0.5048
Epoch 3: saving model to model/Greeting_keypoint/keypoint_classifier.keras
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9080 - loss: 0.3844 - val_accuracy: 0.8570 - val_loss: 0.8120
Epoch 4/1000
57/62 ━━━━━━━━━━━━━━━━━━━━ 0s 966us/step - accuracy: 0.9454 - loss: 0.2286
Epoch 4: saving model to model/Greeting_keypoint/keypoint_classifier.keras
62/62 ━

In [12]:
# Evaluate model
# Model evaluation
val_loss, val_acc = model.evaluate(X_test, y_test, batch_size=128)

21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 903us/step - accuracy: 0.9964 - loss: 0.0175


In [13]:
# Reload model
# Loading the saved model
model = tf.keras.models.load_model(model_save_path)

In [14]:
# Run inference test
# Inference test
predict_result = model.predict(np.array([X_test[0]]))
print(np.squeeze(predict_result))
print(np.argmax(np.squeeze(predict_result)))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
[1.3559469e-10 3.5196335e-11 7.3911832e-09 6.4620588e-12 2.2148644e-08
 3.0384847e-08 9.9999678e-01 3.6169585e-08 7.8626788e-10 3.7325329e-09
 5.4179378e-07 2.3654654e-06 7.3538380e-12 2.4539318e-07 4.8073312e-10]
6


In [16]:
# Save inference model
# Save as a model dedicated to inference
model.save(model_save_path, include_optimizer=False)

In [17]:
# Quantize model
# Transform model (quantization)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quantized_model = converter.convert()

open(tflite_save_path, 'wb').write(tflite_quantized_model)

INFO:tensorflow:Assets written to: /var/folders/0m/xc5njr7x5wd30_5m8c259wr40000gn/T/tmpemti8y93/assets


INFO:tensorflow:Assets written to: /var/folders/0m/xc5njr7x5wd30_5m8c259wr40000gn/T/tmpemti8y93/assets


Saved artifact at '/var/folders/0m/xc5njr7x5wd30_5m8c259wr40000gn/T/tmpemti8y93'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 42), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 15), dtype=tf.float32, name=None)
Captures:
  14649053952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14673986832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14680225568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14648951552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13238347552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14680227680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14668359808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14668358752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14668261504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14668358224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14607829296: Tensor

W0000 00:00:1773078497.348999  166612 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1773078497.349275  166612 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1773078497.362451  166612 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled


24200

In [18]:
# Load TFLite model
interpreter = tf.lite.Interpreter(model_path=tflite_save_path)
interpreter.allocate_tensors()

/Users/ahmad/.pyenv/versions/3.9.6/lib/python3.9/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [19]:
# Read tensor shapes
# Get I / O tensor
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

In [20]:
# Set input sample
interpreter.set_tensor(input_details[0]['index'], np.array([X_test[0]]))

In [21]:
%%time
# Run TFLite inference
# Inference implementation
interpreter.invoke()
tflite_results = interpreter.get_tensor(output_details[0]['index'])

CPU times: user 754 µs, sys: 1.56 ms, total: 2.31 ms
Wall time: 1.98 ms


In [22]:
# Print predictions
print(np.squeeze(tflite_results))
print(np.argmax(np.squeeze(tflite_results)))

[5.8702668e-11 1.7607357e-11 3.3016159e-09 5.3293706e-12 2.7718430e-08
 4.1741369e-08 9.9999857e-01 1.5258252e-08 3.5260653e-10 1.6872880e-09
 1.3757068e-07 6.9540761e-07 2.3230842e-12 4.5501812e-07 2.9570765e-10]
6
